In [ ]:
df = (
    spark
    .read #read
    .option("delimiter", ";") #delimiter 
    .option("header", "true") #show header
    .option("inferSchema", "true") #json to schema auto
    .option("enconding", "ISO-8859-1") #file format to help auto schema
    .csv("data/01.csv")
)

#df config

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
#spark session
spark = SparkSession.builder.master("local[*]").appName("spark_df_gov").getOrCreate()

In [ ]:
df.printSchema()

In [ ]:
df.show(5)

In [ ]:
#To filter, can use dataframe + .where and use functions (F) alias

df.where(F.col("Valor de Compra").isNotNull()).count()


In [ ]:
#using withColumn to change or create column
df = df.withColumn("Valor compra teste",F.col("Valor de Venda"))

In [ ]:
#withColumn to cast column
df_preco = df.select("Estado - Sigla", "Produto", "Valor de Venda", "Unidade de Medida")

#change column, with cast
value_to_replace_regexp = F.regexp_replace(F.col("Valor de Venda"), ",",".")
df_preco = df_preco.withColumn("Valor de Venda", value_to_replace_regexp.cast("float"))

#see new dataType
df_preco.schema["Valor de Venda"].dataType

In [ ]:
#group by

df_preco.groupBy(
    F.col("Estado - Sigla"),
    F.col("Produto")
).agg(
    F.min(F.col("Valor de Venda")).alias("min_value"),
    F.max(F.col("Valor de Venda")).alias("max_value")
).withColumn("difference", (F.col("max_value") - F.col("min_value"))
).orderBy(F.col("difference"), ascending=True).show()